In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

BASE = '/content/drive/MyDrive/disease-rag-project'
folders = [
    'data/breast_cancer', 'data/cad', 'data/diabetes',
    'data/ckd', 'data/nafld', 'data/parkinsons',
    'src', 'configs', 'results', 'notebooks'
]
for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Created:")
for root, dirs, files in os.walk(BASE):
    print(root)

In [ ]:
!pip install -q torch torchvision scikit-learn sentence-transformers transformers rank_bm25 requests gradio torch-geometric pyyaml

In [ ]:
from google.colab import files
uploaded = files.upload()   # select breast_cancer.zip, cad.zip, diabetes.zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content /content/drive/MyDrive -maxdepth 4 -name "*.zip" 2>/dev/null

In [ ]:
!find "{BASE}/data" -maxdepth 4 -type d

In [ ]:
import os

DATA = f'{BASE}/data'
for d in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, d)
    if os.path.isdir(path):
        files = os.listdir(path)
        exts = {os.path.splitext(f)[1].lower() for f in files}
        print(f"{d:15s} {len(files):5d} files   types: {exts}")

In [ ]:
!ls -lh /content/drive/MyDrive/PAPER/*.zip
!unzip -l /content/drive/MyDrive/PAPER/breast_cancer.zip | head -20
!unzip -l /content/drive/MyDrive/PAPER/breast_cancer.zip | tail -3

In [ ]:
import os

for name in ['breast_cancer', 'cad', 'diabetes']:
    os.makedirs(f'/content/data/{name}', exist_ok=True)
    !unzip -o "/content/drive/MyDrive/PAPER/{name}.zip" -d "/content/data/{name}" | tail -3


In [ ]:
for name in ['breast_cancer', 'cad', 'diabetes']:
    root = f'/content/data/{name}'
    counts = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            counts[ext] = counts.get(ext, 0) + 1
    print(f"{name:15s} {counts}")


In [ ]:
import os
from collections import Counter

for name in ['breast_cancer', 'cad', 'diabetes']:
    root = f'/content/data/{name}'
    counts = Counter()
    for dirpath, _, files in os.walk(root):
        if files:
            counts[os.path.relpath(dirpath, root)] += len(files)
    print(f"\n{name}")
    for folder, n in sorted(counts.items()):
        print(f"  {folder:35s} {n}")


In [ ]:
import os, re
import pandas as pd
from collections import Counter
from PIL import Image

rows = []
for disease in ['breast_cancer', 'cad', 'diabetes']:
    for dp, _, fs in os.walk(f'/content/data/{disease}'):
        for f in fs:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                rows.append({'disease': disease,
                             'label': os.path.basename(dp),
                             'path': os.path.join(dp, f),
                             'fname': f})
df = pd.DataFrame(rows)

# Group ID: leading number in breast_cancer filenames; otherwise each file is its own group
df['group'] = df['path']
m = df['disease'] == 'breast_cancer'
df.loc[m, 'group'] = 'bc_' + df.loc[m, 'fname'].str.extract(r'^(\d+)')[0]

print(df.groupby('disease')['group'].nunique())   # breast_cancer: how many distinct cases?

# Image sizes (sampled)
for d, g in df.groupby('disease'):
    sizes = Counter(Image.open(p).size for p in g['path'].sample(min(50, len(g)), random_state=0))
    print(d, sizes.most_common(3))

In [ ]:
bc = df[df.disease == 'breast_cancer']
g = bc.groupby('group').agg(n=('path', 'size'), n_labels=('label', 'nunique'), label=('label', 'first'))

print(g['n'].describe())
print("groups with mixed labels:", (g.n_labels > 1).sum())
print(g.groupby('label').size())

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df['split'] = 'train'
for d, g in df.groupby('disease'):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    _, test_idx = next(sgkf.split(g, g['label'], groups=g['group']))
    df.loc[g.index[test_idx], 'split'] = 'test'

print(pd.crosstab([df.disease, df.split], df.label))
df.to_csv(f'{BASE}/metadata.csv', index=False)

In [ ]:
import hashlib
df['md5'] = df['path'].map(lambda p: hashlib.md5(open(p, 'rb').read()).hexdigest())
print(df.groupby('disease')['md5'].apply(lambda s: s.duplicated().sum()))

In [ ]:
dup = df[df.duplicated('md5', keep=False)]
g = dup.groupby('md5').agg(disease=('disease', 'first'),
                           copies=('path', 'size'),
                           n_labels=('label', 'nunique'),
                           n_splits=('split', 'nunique'))

print(g.groupby('disease').agg(
    hash_groups=('copies', 'size'),
    extra_copies=('copies', lambda s: (s - 1).sum()),
    cross_label=('n_labels', lambda s: (s > 1).sum()),
    cross_split=('n_splits', lambda s: (s > 1).sum())))

# eyeball a few
print(dup.sort_values('md5')[['label', 'split', 'fname']].head(12))

In [ ]:
# drop every copy of images that appear under conflicting labels
conflict = g[g.n_labels > 1].index
df = df[~df['md5'].isin(conflict)]

# keep one copy of the rest
df = df.drop_duplicates('md5', keep='first').reset_index(drop=True)

# re-split
df['split'] = 'train'
for d, sub in df.groupby('disease'):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    _, test_idx = next(sgkf.split(sub, sub['label'], groups=sub['group']))
    df.loc[sub.index[test_idx], 'split'] = 'test'

# per-disease tables are easier to read than one wide crosstab
for d, sub in df.groupby('disease'):
    print(f"\n{d}")
    print(pd.crosstab(sub['split'], sub['label']))

df.to_csv(f'{BASE}/metadata.csv', index=False)

In [ ]:
pairs = dup[dup.md5.isin(conflict)].groupby('md5')['label'].apply(lambda s: ' vs '.join(sorted(s)))
print(pairs.value_counts())

In [ ]:
!pip -q install open_clip_torch

import torch, open_clip, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model = model.to(device).eval()

class DS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return preprocess(Image.open(self.paths[i]).convert('RGB'))

dl = DataLoader(DS(df['path'].tolist()), batch_size=64, num_workers=2)
chunks = []
with torch.no_grad():
    for x in dl:
        f = model.encode_image(x.to(device))
        chunks.append(torch.nn.functional.normalize(f, dim=-1).cpu())
emb = torch.cat(chunks).numpy().astype('float32')

# Save embeddings and metadata together so row order always matches
np.save(f'{BASE}/emb.npy', emb)
df.to_csv(f'{BASE}/metadata.csv', index=False)
print(emb.shape)

In [ ]:
for d in df.disease.unique():
    idx = np.where(df.disease.values == d)[0]
    sub, E = df.iloc[idx], emb[idx]
    is_test = sub.split.values == 'test'

    S = E[is_test] @ E[~is_test].T          # test-vs-train cosine similarity
    max_sim, j = S.max(1), S.argmax(1)

    print(f"\n{d}: test-to-train max similarity percentiles")
    print(np.percentile(max_sim, [50, 90, 99, 100]).round(3))

    close = max_sim > 0.98
    mism = (sub.label.values[is_test][close] != sub.label.values[~is_test][j[close]]).sum()
    print(f"{close.sum()} of {is_test.sum()} test images have a train neighbor >0.98 "
          f"({mism} with a different label)")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'
ZIP_DIR = BASE   # change to the folder where the zips are, e.g. '/content' or f'{BASE}/zips'

!unzip -q "{ZIP_DIR}/breast_cancer.zip" -d "{BASE}/data/breast_cancer"
!unzip -q "{ZIP_DIR}/cad.zip"           -d "{BASE}/data/cad"
!unzip -q "{ZIP_DIR}/diabetes.zip"      -d "{BASE}/data/diabetes"

In [ ]:
!find {BASE}/data -maxdepth 4 -type d

In [ ]:
from google.colab import files
uploaded = files.upload()   # select ckd.zip, nafld.zip, parkinsons.zip

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'
!unzip -q /content/ckd.zip -d {BASE}/data/ckd
!unzip -q /content/NAFLD.zip -d {BASE}/data/nafld
!unzip -q /content/parkinson.zip -d {BASE}/data/parkinsons
!find {BASE}/data -maxdepth 4 -type d

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

config_text = '''
diseases:
  breast_cancer:
    root: data/breast_cancer
    classes: [benign, malignant]
    epochs: 15
    batch_size: 32
    freeze_epochs: 3

  cad:
    root: data/cad
    classes: [normal, abnormal]
    epochs: 30
    batch_size: 16
    freeze_epochs: 5

  diabetes:
    root: data/diabetes
    classes: [Healthy, "Mild DR", "Moderate DR", "Proliferate DR", "Severe DR"]
    epochs: 20
    batch_size: 32
    freeze_epochs: 4

  ckd:
    root: data/ckd
    classes: [Normal, Cyst, Tumor, Stone]
    epochs: 20
    batch_size: 32
    freeze_epochs: 4

  nafld:
    root: data/nafld
    classes: [normal, fatty_liver]
    epochs: 20
    batch_size: 16
    freeze_epochs: 4

  parkinsons:
    root: data/parkinsons
    classes: [healthy, parkinson]
    epochs: 20
    batch_size: 16
    freeze_epochs: 4
'''

with open(f'{BASE}/configs/diseases.yaml', 'w') as f:
    f.write(config_text)

print("Saved config. Contents:")
print(config_text)

In [ ]:
!pip install -q pyyaml

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

data_utils_code = '''
import os
import yaml
from torch.utils.data import Dataset
from PIL import Image

def load_config(base_path):
    with open(os.path.join(base_path, "configs/diseases.yaml")) as f:
        return yaml.safe_load(f)["diseases"]

def find_class_dir(root, class_name):
    direct = os.path.join(root, class_name)
    if os.path.isdir(direct):
        return direct
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() == class_name.lower():
                return os.path.join(dirpath, d)
    raise FileNotFoundError(
        f"Could not find folder \\'{class_name}\\' under \\'{root}\\'. "
        f"Run !find {root} -maxdepth 4 -type d to check real names."
    )

def gather_files(root, classes):
    paths, labels = [], []
    for i, c in enumerate(classes):
        cdir = find_class_dir(root, c)
        print(f"  using \\'{c}\\' -> {cdir}")
        for fn in os.listdir(cdir):
            if fn.lower().endswith((".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(cdir, fn))
                labels.append(i)
    return paths, labels

class ImgListDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.tf(img), self.labels[idx]
'''

with open(f'{BASE}/src/data_utils.py', 'w') as f:
    f.write(data_utils_code)

print("data_utils.py created.")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

models_code = '''
import torch
import torch.nn as nn
from torchvision import models, transforms

def get_transforms(img_size=224):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf

def build_resnet18(num_classes, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, num_classes)
    return model

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    return model
'''

with open(f'{BASE}/src/models.py', 'w') as f:
    f.write(models_code)

print("models.py created.")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

train_code = '''
import os
import csv
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from data_utils import load_config, gather_files, ImgListDataset
from models import get_transforms, build_resnet18, unfreeze_all

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_one_disease(base_path, disease_name):
    config = load_config(base_path)
    cfg = config[disease_name]
    root = os.path.join(base_path, cfg["root"])
    classes = cfg["classes"]
    epochs = cfg["epochs"]
    batch_size = cfg["batch_size"]
    freeze_epochs = cfg["freeze_epochs"]

    print(f"=== Training {disease_name} on {DEVICE} ===")
    paths, labels = gather_files(root, classes)
    print(f"Total images: {len(paths)}")

    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=0.2, stratify=labels, random_state=42
    )

    train_tf, eval_tf = get_transforms()
    train_ds = ImgListDataset(train_paths, train_labels, train_tf)
    val_ds = ImgListDataset(val_paths, val_labels, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = build_resnet18(num_classes=len(classes), freeze_backbone=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    results_dir = os.path.join(base_path, "results", disease_name)
    os.makedirs(results_dir, exist_ok=True)
    log_path = os.path.join(results_dir, "train_log.csv")
    best_ckpt = os.path.join(results_dir, "best_model.pt")

    best_val_acc = 0.0
    with open(log_path, "w", newline="") as logf:
        writer = csv.writer(logf)
        writer.writerow(["epoch", "train_loss", "val_acc"])

        for epoch in range(1, epochs + 1):
            if epoch == freeze_epochs + 1:
                model = unfreeze_all(model)
                print(f"  -> unfroze backbone at epoch {epoch}")

            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
            )

            model.train()
            running_loss = 0.0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                optimizer.zero_grad()
                out = model(imgs)
                loss = criterion(out, lbls)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * imgs.size(0)
            train_loss = running_loss / len(train_ds)

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                    out = model(imgs)
                    preds = out.argmax(dim=1)
                    correct += (preds == lbls).sum().item()
                    total += lbls.size(0)
            val_acc = correct / total

            print(f"  epoch {epoch}/{epochs} | loss={train_loss:.4f} | val_acc={val_acc:.4f}")
            writer.writerow([epoch, train_loss, val_acc])
            logf.flush()

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save({
                    "model_state": model.state_dict(),
                    "classes": classes,
                    "val_acc": val_acc,
                }, best_ckpt)

    print(f"Best val_acc for {disease_name}: {best_val_acc:.4f}")
    print(f"Checkpoint saved to {best_ckpt}")
    return best_val_acc
'''

with open(f'{BASE}/src/train.py', 'w') as f:
    f.write(train_code)

print("train.py created.")

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import train
importlib.reload(train)
from train import train_one_disease

acc = train_one_disease(BASE, 'breast_cancer')

In [ ]:
diseases_to_train = ['breast_cancer', 'cad', 'diabetes', 'ckd', 'nafld', 'parkinsons']
all_results = {}

for d in diseases_to_train:
    try:
        all_results[d] = train_one_disease(BASE, d)
    except Exception as e:
        print(f"!! {d} failed: {e}")
        all_results[d] = None

print("\n=== Summary ===")
for d, acc in all_results.items():
    print(f"{d}: {acc}")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

evaluate_code = '''
import os
import torch
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

from data_utils import load_config, gather_files, ImgListDataset
from models import get_transforms, build_resnet18
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate_disease(base_path, disease_name):
    config = load_config(base_path)
    cfg = config[disease_name]
    root = os.path.join(base_path, cfg["root"])
    classes = cfg["classes"]

    ckpt_path = os.path.join(base_path, "results", disease_name, "best_model.pt")
    if not os.path.exists(ckpt_path):
        print(f"No checkpoint found for {disease_name} at {ckpt_path}, skipping.")
        return

    paths, labels = gather_files(root, classes)
    _, val_paths, _, val_labels = train_test_split(
        paths, labels, test_size=0.2, stratify=labels, random_state=42
    )

    _, eval_tf = get_transforms()
    val_ds = ImgListDataset(val_paths, val_labels, eval_tf)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

    model = build_resnet18(num_classes=len(classes), freeze_backbone=False).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)
            preds = out.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(lbls.tolist())

    print(f"\\n=== {disease_name} Evaluation ===")
    print(classification_report(all_labels, all_preds, target_names=classes))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{disease_name} Confusion Matrix")
    fig.colorbar(im)
    plt.tight_layout()

    fig_path = os.path.join(base_path, "results", disease_name, "confusion_matrix.png")
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved confusion matrix to {fig_path}")
'''

with open(f'{BASE}/src/evaluate.py', 'w') as f:
    f.write(evaluate_code)

print("evaluate.py created.")

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import evaluate
importlib.reload(evaluate)
from evaluate import evaluate_disease

diseases = ['breast_cancer', 'cad', 'diabetes', 'ckd', 'nafld', 'parkinsons']
for d in diseases:
    evaluate_disease(BASE, d)

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'
for d in ['breast_cancer', 'cad', 'diabetes', 'ckd', 'nafld', 'parkinsons']:
    path = f'{BASE}/results/{d}/best_model.pt'
    print(d, "->", "OK" if os.path.exists(path) else "MISSING")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

predict_code = '''
import os
import torch
from PIL import Image

from data_utils import load_config
from models import get_transforms, build_resnet18

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_model_cache = {}

def load_disease_model(base_path, disease_name):
    if disease_name in _model_cache:
        return _model_cache[disease_name]

    config = load_config(base_path)
    cfg = config[disease_name]
    classes = cfg["classes"]

    ckpt_path = os.path.join(base_path, "results", disease_name, "best_model.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"No checkpoint at {ckpt_path}")

    model = build_resnet18(num_classes=len(classes), freeze_backbone=False).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    _model_cache[disease_name] = (model, classes)
    return model, classes

def predict_image(base_path, disease_name, image_path):
    model, classes = load_disease_model(base_path, disease_name)
    _, eval_tf = get_transforms()

    img = Image.open(image_path).convert("RGB")
    tensor = eval_tf(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().squeeze()

    pred_idx = probs.argmax().item()
    return {
        "disease": disease_name,
        "predicted_class": classes[pred_idx],
        "confidence": round(probs[pred_idx].item(), 4),
        "all_probs": {c: round(p.item(), 4) for c, p in zip(classes, probs)},
        "image_path": image_path,
    }
'''

with open(f'{BASE}/src/predict.py', 'w') as f:
    f.write(predict_code)

print("predict.py created.")

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import predict
importlib.reload(predict)
from predict import predict_image

# pick any real image path from your data folder to sanity check
import glob
sample_imgs = glob.glob(f'{BASE}/data/breast_cancer/**/*.jpg', recursive=True) + \
              glob.glob(f'{BASE}/data/breast_cancer/**/*.png', recursive=True)

if sample_imgs:
    result = predict_image(BASE, 'breast_cancer', sample_imgs[0])
    print(result)
else:
    print("No sample images found — check your data/breast_cancer folder structure.")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

router_code = '''
DISEASE_KEYWORDS = {
    "breast_cancer": ["breast", "mammogram", "mammography", "lump"],
    "cad": ["cad", "coronary", "artery disease", "cardiac", "heart"],
    "diabetes": ["diabetic retinopathy", "retina", "retinal", "diabetes eye"],
    "ckd": ["kidney", "renal", "ckd", "cyst", "kidney stone"],
    "nafld": ["liver", "fatty liver", "nafld", "hepatic"],
    "parkinsons": ["parkinson", "tremor", "spiral drawing", "wave drawing"],
}

def route_to_disease(user_query_text):
    """
    Simple keyword-based router. Returns the best-matching disease key,
    or None if nothing matches (caller should ask the user to clarify
    or specify the image type explicitly).
    """
    query_lower = user_query_text.lower()
    scores = {}
    for disease, keywords in DISEASE_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in query_lower)
        if score > 0:
            scores[disease] = score

    if not scores:
        return None

    return max(scores, key=scores.get)
'''

with open(f'{BASE}/src/router.py', 'w') as f:
    f.write(router_code)

print("router.py created.")

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import router
importlib.reload(router)
from router import route_to_disease

test_queries = [
    "Patient has a retinal scan showing signs of diabetic damage",
    "This is a kidney CT scan, checking for a cyst",
    "Ultrasound of the liver, checking for fatty deposits",
    "Spiral drawing test for tremor assessment",
]

for q in test_queries:
    print(f"'{q}' -> {route_to_disease(q)}")

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

agent_code = '''
import os
import sys

from router import route_to_disease
from predict import predict_image, load_disease_model

class MedicalAgentError(Exception):
    pass

def run_agent(base_path, query_text, image_path, disease_override=None):
    """
    End-to-end pipeline:
      1. Route query_text -> disease (unless disease_override given)
      2. Run the matching trained classifier on image_path
      3. Return a structured result ready to hand to the RAG/report layer

    disease_override: pass a disease key directly (e.g. 'ckd') to skip
    the keyword router entirely.
    """
    if disease_override:
        disease = disease_override
    else:
        disease = route_to_disease(query_text)

    if disease is None:
        raise MedicalAgentError(
            "Could not determine which disease pipeline to use from the query. "
            "Please specify the image type explicitly (breast_cancer, cad, "
            "diabetes, ckd, nafld, or parkinsons)."
        )

    if not os.path.exists(image_path):
        raise MedicalAgentError(f"Image not found at {image_path}")

    try:
        prediction = predict_image(base_path, disease, image_path)
    except FileNotFoundError as e:
        raise MedicalAgentError(f"Model not trained for {disease}: {e}")

    return {
        "disease": disease,
        "query": query_text,
        "prediction": prediction,
    }

def format_agent_result(result):
    pred = result["prediction"]
    lines = [
        f"Disease pipeline : {result['disease']}",
        f"Predicted class  : {pred['predicted_class']}",
        f"Confidence       : {pred['confidence']:.1%}",
        f"All probabilities:",
    ]
    for cls, prob in pred["all_probs"].items():
        lines.append(f"  {cls}: {prob:.4f}")
    return "\\n".join(lines)
'''

with open(f'{BASE}/src/agent.py', 'w') as f:
    f.write(agent_code)

print("agent.py created.")

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import agent
importlib.reload(agent)
from agent import run_agent, format_agent_result

import glob
sample_imgs = glob.glob(f'{BASE}/data/ckd/**/*.jpg', recursive=True) + \
              glob.glob(f'{BASE}/data/ckd/**/*.png', recursive=True)

if sample_imgs:
    result = run_agent(
        base_path=BASE,
        query_text="Patient CT scan, checking for kidney abnormalities",
        image_path=sample_imgs[0],
    )
    print(format_agent_result(result))
else:
    print("No sample CKD images found — check data/ckd folder structure.")

In [ ]:
if sample_imgs:
    result = run_agent(
        base_path=BASE,
        query_text="ignored when override is set",
        image_path=sample_imgs[0],
        disease_override="ckd",
    )
    print(format_agent_result(result))

In [ ]:
BASE = '/content/drive/MyDrive/disease-rag-project'

report_pipeline_code = '''
import os
import sys
import requests

from agent import run_agent, format_agent_result

OLLAMA_URL = "http://localhost:11434/api/generate"

def generate_report(base_path, query_text, image_path, disease_override=None):
    """
    1. Run the classification agent
    2. Build a prompt with the prediction result
    3. Call Ollama/Mistral to generate a medical-style report
    """
    result = run_agent(base_path, query_text, image_path, disease_override)
    agent_summary = format_agent_result(result)

    prompt = f"""You are a medical AI assistant. Based on the following image classification result, 
write a concise clinical-style report. Include the finding, confidence level, and a brief recommendation.

Classification Result:
{agent_summary}

Patient query: {query_text}

Please write a professional medical report based on this analysis."""

    try:
        r = requests.post(
            OLLAMA_URL,
            json={"model": "mistral", "prompt": prompt, "stream": False},
            timeout=120,
        )
        r.raise_for_status()
        report_text = r.json()["response"]
    except Exception as e:
        report_text = f"[Ollama unavailable: {e}]\\n\\nFallback summary:\\n{agent_summary}"

    result["report"] = report_text
    return result
'''

with open(f'{BASE}/src/report_pipeline.py', 'w') as f:
    f.write(report_pipeline_code)

print("report_pipeline.py created.")

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
subprocess.Popen(['ollama', 'serve'])
import time
time.sleep(5)
print("Ollama server starting...")

In [ ]:
!ollama pull mistral

In [ ]:
import requests
try:
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": "Say hello in one sentence.", "stream": False},
        timeout=60,
    )
    print(r.json()["response"])
except Exception as e:
    print("Ollama not reachable:", e)

In [ ]:
import sys
BASE = '/content/drive/MyDrive/disease-rag-project'
sys.path.append(f'{BASE}/src')

import importlib
import report_pipeline
importlib.reload(report_pipeline)
from report_pipeline import generate_report

import glob
sample_imgs = glob.glob(f'{BASE}/data/ckd/**/*.jpg', recursive=True) + \
              glob.glob(f'{BASE}/data/ckd/**/*.png', recursive=True)

if sample_imgs:
    result = generate_report(
        base_path=BASE,
        query_text="Patient CT scan, checking for kidney abnormalities",
        image_path=sample_imgs[0],
    )
    print("=== FINAL REPORT ===")
    print(result["report"])
else:
    print("No sample images found.")